In [10]:
import os
import sqlite3
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import pandas as pd
import plotly.express as px

In [11]:
# Initialize the Dash app
app = dash.Dash(__name__)

# Define the layout of the app
app.layout = html.Div([
    html.H1("Incident Data by County and Year"),
    dcc.Graph(id='incident-bar-chart')
])

In [19]:
# Define a function to fetch data and create the Plotly figure
def get_figure():
    # Connect to the SQLite database
    conn = sqlite3.connect(os.path.join(os.getcwd(), 'data/crash_data.db'))

    # Define the query
    query = '''
    SELECT
        County,
        strftime("%Y", CollisionDate) AS CollisionYear,
        COUNT(*) AS IncidentCount
    FROM
        collision_incidents
    GROUP BY
        County, CollisionYear
    ORDER BY
        County, CollisionYear;
    '''

    # Execute the query and fetch the results into a DataFrame
    df = pd.read_sql_query(query, conn)

    # Close the database connection
    conn.close()

    print(df)

    # Create the bar chart using Plotly Express
    fig = px.bar(df, x='CollisionYear', y='IncidentCount', color='County', barmode='group',
                 labels={'CollisionYear': 'Year', 'IncidentCount': 'Number of Incidents'},
                 title='Number of Incidents by County and Year')

    return fig

In [14]:
# Define the callback to update the graph
@app.callback(
    Output('incident-bar-chart', 'figure'),
    [Input('incident-bar-chart', 'id')]  # Dummy input to trigger the callback
)
def update_graph(_):
    return get_figure()

In [16]:
# Run the app on a different port (e.g., 8051)
if __name__ == '__main__':
    app.run_server(debug=True, port=8051)
